# TO:DO 
## check: https://www.nature.com/articles/nature17995 cumulative effect of mutations

Fitness score is cholesterol accumulation.\
This is the article for this dataset: https://pubmed.ncbi.nlm.nih.gov/35190686/ \
Can I predict severity scores of real patients using the model trained on fitness scores?

## Why genomic model?

* We lose information if we use protein sequences.
* 3 **synonomous** substitutions had deleterious effect on function score
* 10 positions had different SPE Classification depending on nucleotide.

## Plan

1. Train model on HEK dataset.
2. Show model generalization across different parts of the protein
3. Show model generalization on different cell line (make sure positions are different and there is no data leakage!)
4. Use fitness score model to predict severity scores from genome??? But sounds very cool, need to think how to bridge these two! Maybe I can fine tune foundation model on dataset and then produce embeddings and then embeddigns x learnable weight for severity score.

## Bimodal distribution 
1. try first simple regression if quality is poor
2. then mixture of experts: split samples to 2 groups and then train separate regressors for each group.

## Embeddings

If you want to use all your data for embeddings eventually (e.g. for a final production model), the workflow is:

Do train/test split
Fine-tune on train, evaluate on test to get honest performance metrics
Once satisfied, retrain final model on full dataset — but you report the performance from step 2, not from the full-data model

Why I plan to use RNA and I don't want to use DNA - Length of DNA is 62119

Clinvar as additional validation

In [5]:
import pandas as pd

# CDS is a position within coding sequence
df = pd.read_excel('data/NPC1_mut_fitness_scores.xlsx', header=1)
df.head(2)

,Protein Annotation,Wild type Base,Edited Base,CDS,Consequence,SPE Classification,Function Score,id,start,end,reference_base,alternate_base,refseq_id,Clinvar_SIG,CADD_phred,Unadjusted Function Score
0,L1027L,T,A,3081,synonymous,Functional,0.954726,chr18-23536836-23536837-A-T,23536836,23536837,A,T,NPC1:NM_000271:exon21:c.T3081A:p.L1027L:Select,NaN,NaN,0.188709
1,L1027L,T,C,3081,synonymous,Functional,0.999139,chr18-23536836-23536837-A-G,23536836,23536837,A,G,NPC1:NM_000271:exon21:c.T3081C:p.L1027L:Select,NaN,NaN,0.008803


I1061T (wild type T -> edited base C), position is end (23536736) NC_000018.10:g.23536736A>G \
Reference base - reference base in the DNA (reverse strand)!\
Wild type base - reference base in the RNA\
Check here: df[~df['Clinvar_SIG'].isna()] - first one!

In [15]:
numbers = [int(x[1:-1]) if isinstance(x,str) else x
           for x in df['Protein Annotation'].unique()]
min(numbers), max(numbers)

(347, 1190)

In [29]:
print(f"The size of the dataframe {df.shape}")


print(f"There are {df[df['Consequence'] == 'splice region'].shape[0]} variants in the splice region. They will be excluded from the dataset for modeling, as we will focus on coding sequence.")
df = df[~(df['Consequence'] == 'splice region')]
sample_number = df.shape[0]
#df.to_csv('output/df_preprocessed.csv')
df['CDS'].min(), df['CDS'].max()

The size of the dataframe (978, 16)
There are 9 variants in the splice region. They will be excluded from the dataset for modeling, as we will focus on coding sequence.


(1040, 3568)

In [21]:
import requests
from Bio import SeqIO
from io import StringIO
import numpy as np

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_000018.10&rettype=fasta&retmode=text"
response = requests.get(url)

record_DNA  = SeqIO.read(StringIO(response.text), "fasta")
seq = record_DNA.seq

In [44]:
end_pos = 23536837
ind = end_pos - 1
print(f"Length of DNA NPC1: {len(seq)} nucleotides")
print(f"Genomic position: {ind}")
print(f"Reference nucleotide: {seq[ind]}")


window_size = 8192

def mutate_seq(seq, pos, new_base):
    ind = pos - 1
    assert df['reference_base'][i] == seq[ind], f"Mismatch at ind={ind}"

    ref_seq = str(seq)
    start = max(0, ind - window_size//2)
    end = min(window_size//2+1 + ind, len(ref_seq))
    ref_seq = seq[start:end]
    mut_seq = seq[start:ind] + new_base + seq[ind+1:end] 
    return ref_seq, mut_seq

new_seqs = []
ref_seqs = []

for i in (df.index):
    pos = df['end'][i]
    new_base = df['alternate_base'][i]
    new_seq, ref_seq = mutate_seq(seq, pos, new_base)
    new_seqs.append(new_seq)
    ref_seqs.append(ref_seq)

new_seqs, ref_seqs = np.array(new_seqs), np.array(ref_seqs)
assert sample_number == new_seqs.shape[0] == ref_seqs.shape[0]

new_seqs_rc = np.array([str(Seq(''.join(s)).reverse_complement()) for s in new_seqs])
ref_seqs_rc = np.array([str(Seq(''.join(s)).reverse_complement()) for s in ref_seqs])

np.save("output/mut_seq_DNA.npy", new_seqs_rc)
np.save("output/ref_seq_DNA.npy", ref_seqs_rc)


Length of DNA NPC1: 80373285 nucleotides
Genomic position: 23536836
Reference nucleotide: A


In [48]:
print(f'Reference, SNV 0: ...{new_seq[4082:4112]}...')
print(f'Variant, SNV 0:   ...{ref_seq[4082:4112]}...')

Reference, SNV 0: ...AAATATCTGCTGCACCAGGGAATCATTGTT...
Variant, SNV 0:   ...AAATATCTGCTGCAACAGGGAATCATTGTT...
